# Conditional Aggregation

## Overview
Conditional aggregation computes different aggregates for different subsets of data **within a single query**, without needing multiple subqueries or joins. The pattern is: wrap a `CASE WHEN` expression inside an aggregate function.

**Core pattern:**
```sql
-- Count rows matching a condition
COUNT(CASE WHEN condition THEN 1 END)

-- Sum only the values matching a condition
SUM(CASE WHEN condition THEN amount ELSE 0 END)

-- Average only the values matching a condition
AVG(CASE WHEN condition THEN amount END)   -- NULLs excluded automatically
```

**Why use this instead of separate queries?**
- One pass over the data instead of N passes
- Results are in one row/table — easy to compare side by side
- Forms the basis of SQL pivot tables (categories become columns)
- Much more readable than self-joins or subqueries for cross-tab reports

**PostgreSQL shorthand:** `FILTER (WHERE condition)` clause on any aggregate function:
```sql
COUNT(*) FILTER (WHERE txn_type = 'Deposit')
SUM(amount) FILTER (WHERE amount > 0)
```
Both approaches produce the same result. `FILTER` is cleaner but PostgreSQL-only.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
    segment TEXT, province TEXT
);
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT,
    province TEXT, opened_date TEXT, status TEXT
);
CREATE TABLE transactions (
    txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT,
    txn_type TEXT, amount REAL, category TEXT, flagged INTEGER
);
CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY, customer_id INTEGER, loan_type TEXT,
    principal REAL, rate_pct REAL, term_months INTEGER,
    issued_date TEXT, status TEXT
);
INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),(2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),(4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),(6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),(8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),(10,'Marcus','Okafor','Retail','ON');
INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),
  (102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),
  (104,2,'Investment','BC','2021-01-10','Active'),
  (105,3,'Chequing','ON','2018-05-20','Active'),
  (106,3,'Investment','ON','2018-05-20','Active'),
  (107,3,'Savings','ON','2022-11-01','Active'),
  (108,4,'Chequing','NB','2015-09-30','Active'),
  (109,4,'Loan','NB','2020-04-01','Closed'),
  (110,5,'Chequing','BC','2021-06-15','Active'),
  (111,5,'Savings','BC','2021-06-15','Suspended'),
  (112,6,'Chequing','AB','2017-11-22','Active'),
  (113,7,'Investment','ON','2016-03-08','Active'),
  (114,7,'Chequing','ON','2016-03-08','Active'),
  (115,8,'Chequing','QC','2023-01-05','Active'),
  (116,9,'Chequing','BC','2022-08-19','Active'),
  (117,9,'Investment','BC','2022-08-19','Active'),
  (118,10,'Chequing','ON','2020-12-01','Active'),
  (119,10,'Savings','ON','2020-12-01','Active');
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),
  (1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),
  (1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),
  (1006,102,'2023-01-15','Deposit',500.00,'Transfer',0),
  (1007,102,'2023-03-10','Interest',12.50,'Interest',0),
  (1008,103,'2023-01-08','Deposit',15000.00,'Payroll',0),
  (1009,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1010,103,'2023-02-08','Deposit',15000.00,'Payroll',0),
  (1011,104,'2023-01-31','Deposit',5000.00,'Transfer',0),
  (1012,105,'2023-01-10','Deposit',22000.00,'Payroll',0),
  (1013,105,'2023-01-28','Withdrawal',-8500.00,'Rent',0),
  (1014,105,'2023-02-10','Deposit',22000.00,'Payroll',0),
  (1015,106,'2023-02-01','Deposit',50000.00,'Investment',0),
  (1016,108,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1017,108,'2023-01-19','Withdrawal',-700.00,'Rent',0),
  (1018,108,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1019,110,'2023-01-15','Deposit',8500.00,'Payroll',0),
  (1020,110,'2023-02-01','Withdrawal',-2100.00,'Rent',0),
  (1021,112,'2023-01-20','Deposit',3800.00,'Payroll',0),
  (1022,112,'2023-02-10','Fee',-25.00,'Fee',1),
  (1023,113,'2023-01-05','Deposit',75000.00,'Investment',0),
  (1024,114,'2023-01-12','Deposit',12000.00,'Payroll',0),
  (1025,114,'2023-02-12','Deposit',12000.00,'Payroll',0),
  (1026,115,'2023-01-10','Deposit',2800.00,'Payroll',0),
  (1027,115,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1028,116,'2023-01-18','Deposit',9200.00,'Payroll',0),
  (1029,117,'2023-02-05','Deposit',10000.00,'Investment',0),
  (1030,118,'2023-01-22','Deposit',4500.00,'Payroll',0),
  (1031,118,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1032,119,'2023-02-20','Interest',18.75,'Interest',0),
  (1033,101,'2023-03-05','Deposit',4200.00,'Payroll',NULL),
  (1034,103,'2023-03-08','Deposit',15000.00,'Payroll',0),
  (1035,112,'2023-03-15','Withdrawal',-450.00,'Groceries',1);
INSERT INTO loans VALUES
  (201,1,'Personal',15000.00,7.5,36,'2021-06-01','Current'),
  (202,2,'Mortgage',480000.00,4.2,300,'2020-07-15','Current'),
  (203,3,'HELOC',100000.00,6.1,60,'2019-02-28','Current'),
  (204,4,'Auto',28000.00,5.9,60,'2020-04-01','Paid Off'),
  (205,5,'Mortgage',390000.00,4.8,300,'2021-06-20','Current'),
  (206,6,'Personal',8000.00,9.2,24,'2022-03-10','Delinquent'),
  (207,7,'Mortgage',750000.00,3.9,300,'2018-09-01','Current'),
  (208,8,'Auto',22000.00,6.5,48,'2023-01-15','Current'),
  (209,9,'Personal',12000.00,8.1,36,'2022-08-25','Current'),
  (210,10,'Auto',35000.00,5.4,60,'2021-03-01','Current'),
  (211,3,'Personal',25000.00,6.8,48,'2020-11-15','Paid Off'),
  (212,7,'HELOC',200000.00,5.5,60,'2022-06-01','Current');
""")
conn.commit()
print('Finance schema ready.')


---
## Counting subsets within a group

In [ ]:
# Per-account: break down transaction counts by type in one row
print('=== Transaction type breakdown per account (conditional COUNT) ===')
q = """
SELECT
    account_id,
    COUNT(*)                                           AS total_txns,
    COUNT(CASE WHEN txn_type = 'Deposit'    THEN 1 END) AS deposits,
    COUNT(CASE WHEN txn_type = 'Withdrawal' THEN 1 END) AS withdrawals,
    COUNT(CASE WHEN txn_type = 'Fee'        THEN 1 END) AS fees,
    COUNT(CASE WHEN txn_type = 'Interest'   THEN 1 END) AS interest_credits,
    COUNT(CASE WHEN COALESCE(flagged,0) = 1 THEN 1 END) AS flagged_count
FROM    transactions
GROUP BY account_id
ORDER BY total_txns DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Summing subsets — credits vs debits

In [ ]:
# Split total amount into credits and debits in one row per account
print('=== Credits vs debits per account ===')
q = """
SELECT
    account_id,
    ROUND(SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END), 2)   AS total_credits,
    ROUND(SUM(CASE WHEN amount < 0 THEN amount ELSE 0 END), 2)   AS total_debits,
    ROUND(SUM(amount), 2)                                         AS net_position,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 2)          AS avg_credit,
    ROUND(AVG(CASE WHEN amount < 0 THEN amount END), 2)          AS avg_debit
FROM    transactions
GROUP BY account_id
ORDER BY net_position DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Pivot table — categories become columns

In [ ]:
# Pivot: one row per province, columns for each loan type's total principal
print('=== Loan principal pivot: provinces as rows, loan types as columns ===')
q = """
SELECT
    c.province,
    COUNT(l.loan_id)                                                   AS total_loans,
    ROUND(SUM(CASE WHEN l.loan_type = 'Mortgage'  THEN l.principal END), 2) AS mortgage_total,
    ROUND(SUM(CASE WHEN l.loan_type = 'Auto'      THEN l.principal END), 2) AS auto_total,
    ROUND(SUM(CASE WHEN l.loan_type = 'Personal'  THEN l.principal END), 2) AS personal_total,
    ROUND(SUM(CASE WHEN l.loan_type = 'HELOC'     THEN l.principal END), 2) AS heloc_total,
    ROUND(SUM(l.principal), 2)                                         AS grand_total
FROM    loans AS l
INNER JOIN customers AS c ON l.customer_id = c.customer_id
GROUP BY c.province
ORDER BY grand_total DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print('PostgreSQL CROSSTAB() (tablefunc extension) can generate pivots dynamically,')
print('but CASE WHEN is portable and explicit — preferred for a known set of categories.')


---
## Monthly trend with conditional columns

In [ ]:
# Monthly summary: deposits, withdrawals, fees, flagged — all in one row per month
print('=== Monthly activity breakdown ===')
q = """
SELECT
    STRFTIME('%Y-%m', txn_date)                                         AS month,
    COUNT(*)                                                            AS total_txns,
    COUNT(CASE WHEN txn_type = 'Deposit'    THEN 1 END)                AS deposit_count,
    COUNT(CASE WHEN txn_type = 'Withdrawal' THEN 1 END)                AS withdrawal_count,
    ROUND(SUM(CASE WHEN txn_type = 'Deposit'    THEN amount ELSE 0 END), 2) AS deposit_amt,
    ROUND(SUM(CASE WHEN txn_type = 'Withdrawal' THEN amount ELSE 0 END), 2) AS withdrawal_amt,
    COUNT(CASE WHEN COALESCE(flagged,0) = 1     THEN 1 END)            AS flagged_txns,
    ROUND(100.0 * COUNT(CASE WHEN COALESCE(flagged,0) = 1 THEN 1 END)
               / COUNT(*), 1)                                           AS flagged_pct
FROM    transactions
GROUP BY month
ORDER BY month
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## PostgreSQL FILTER clause

In [ ]:
# PostgreSQL FILTER — cleaner syntax, same results as CASE WHEN
print('=== PostgreSQL FILTER clause (reference — not runnable in SQLite) ===')
print("""
SELECT
    account_id,
    COUNT(*)                                    AS total_txns,
    COUNT(*) FILTER (WHERE txn_type = 'Deposit')      AS deposits,
    COUNT(*) FILTER (WHERE txn_type = 'Withdrawal')   AS withdrawals,
    SUM(amount) FILTER (WHERE amount > 0)             AS total_credits,
    SUM(amount) FILTER (WHERE amount < 0)             AS total_debits,
    AVG(amount) FILTER (WHERE txn_type = 'Deposit')   AS avg_deposit
FROM transactions
GROUP BY account_id;

-- FILTER also works with window functions in PostgreSQL:
SELECT
    txn_date,
    COUNT(*) FILTER (WHERE txn_type = 'Deposit')
        OVER (ORDER BY txn_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
    AS rolling_7day_deposit_count
FROM transactions;
""")

conn.close()


---
## Common Pitfalls

**1. Using `ELSE 0` in AVG distorts the average**
`AVG(CASE WHEN txn_type = 'Deposit' THEN amount ELSE 0 END)` includes all non-deposit rows as zeros in the average, making the result much lower than the true average deposit amount. For AVG, omit the ELSE clause so non-matching rows become NULL and are excluded: `AVG(CASE WHEN txn_type = 'Deposit' THEN amount END)`.

**2. Using `ELSE 0` in COUNT is harmless but unnecessary**
`COUNT(CASE WHEN condition THEN 1 ELSE 0 END)` counts all rows because both 1 and 0 are non-NULL. Use `COUNT(CASE WHEN condition THEN 1 END)` — the ELSE NULL is implicit and correctly skips non-matching rows.

**3. NULL flags treated as non-matching in CASE WHEN**
`CASE WHEN flagged = 1 THEN 1 END` returns NULL when `flagged IS NULL` (not 0). This is usually correct (NULLs are excluded from COUNT), but if NULLs mean "not flagged" in your data, use `COALESCE(flagged, 0)` in the condition.

**4. Pivot columns require knowing the categories upfront**
The `CASE WHEN loan_type = 'Mortgage'` approach only works when you know all possible category values at query-write time. If new loan types are added, the query silently omits them. PostgreSQL's `CROSSTAB()` or application-level pivoting with pandas handles dynamic categories.

**5. Forgetting to include the non-conditional total for rate calculations**
`COUNT(CASE WHEN flagged = 1 THEN 1 END) / COUNT(*)` — if you forget to include `COUNT(*)` as a separate column, you can't compute the flagged rate. Always include both the conditional count and the total in the same SELECT.

**6. Conditional SUM vs conditional COUNT give different things**
`SUM(CASE WHEN txn_type = 'Deposit' THEN amount END)` is the total dollar value of deposits. `COUNT(CASE WHEN txn_type = 'Deposit' THEN 1 END)` is the number of deposit transactions. These answer different questions — choose based on what you need to measure.


---
*sql_methods_library - Samantha McGarrigle*